# 03 — Baseline Training

**Goal:** Train a minimal semantic segmentation CNN from scratch to verify the full pipeline runs end-to-end.

---

## What model are we using?

This notebook uses a **very simple 3-layer CNN** as a placeholder.  
Its purpose is to confirm that:
- Data loading works correctly.
- Tensor shapes are correct at every step.
- The loss goes down (even slightly) after one epoch.
- Checkpointing saves and restores weights.

This model will **not** produce good segmentations — that is expected.  
Once the pipeline is verified, replace the model with U-Net, FCN, or DeepLabv3.

> **Prerequisite:** Run notebooks 00–02 first and ensure `pairs` is populated.

In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from src.data_paths import RAW_DATA_DIR, MODELS_DIR, ensure_project_dirs
from src.dataset import AI4MarsDataset, find_image_files, find_mask_files, build_pairs_by_stem
from src.train_utils import get_device, save_checkpoint, train_one_epoch, evaluate

ensure_project_dirs()

print(f"PyTorch version: {torch.__version__}")

All project directories are ready.
PyTorch version: 2.12.1+cpu


## Step 1 — Load Pairs

In [2]:
DATA_ROOT = RAW_DATA_DIR  # adjust if needed

image_files = find_image_files(DATA_ROOT)
mask_files  = find_mask_files(DATA_ROOT)
pairs       = build_pairs_by_stem(image_files, mask_files)

print(f"Total pairs (all schemes): {len(pairs)}")

# Restrict baseline training to NAV labels only.
# M2020_GEO masks contain different class IDs (e.g. 20, 30, 50) that are
# incompatible with a 4-class NAV setup (NUM_CLASSES=4).
nav_pairs = [(img, msk) for img, msk in pairs if "M2020_GEO" not in str(msk)]
pairs = nav_pairs
print(f"Total pairs (NAV only)   : {len(pairs)}")

if len(pairs) == 0:
    raise RuntimeError(
        "No NAV image/mask pairs found. "
        "Ensure AI4Mars data is extracted under data/raw/ and labels are present."
    )


Total pairs (all schemes): 46460
Total pairs (NAV only)   : 37958


## Step 2 — Hyperparameters

In [3]:
IMAGE_SIZE   = (256, 256)   # (width, height) — PIL convention
BATCH_SIZE   = 4
NUM_CLASSES  = 4            # soil, bedrock, sand, big_rock
IGNORE_INDEX = 255          # unlabeled pixels in AI4Mars masks
LEARNING_RATE = 1e-3
NUM_EPOCHS   = 3            # quick quality check for weighted + pretrained baseline
VAL_SPLIT    = 0.1          # 10% of data held out for validation

## Step 3 — Create Train/Val Split and DataLoaders

In [8]:
# Reproducible split
total = len(pairs)
val_size   = max(1, int(total * VAL_SPLIT))
train_size = total - val_size

# Create the full dataset first so we can split by index
full_dataset = AI4MarsDataset(pairs, image_size=IMAGE_SIZE)
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")

# DataLoader performance knobs:
# - Windows/CPU often plateaus with too many workers due to process overhead.
cpu_count = os.cpu_count() or 2
if torch.cuda.is_available():
    NUM_WORKERS = max(2, min(8, cpu_count - 1))
else:
    NUM_WORKERS = min(4, max(1, cpu_count // 2))

PIN_MEMORY = torch.cuda.is_available()

print(f"DataLoader num_workers: {NUM_WORKERS}")
print(f"DataLoader pin_memory : {PIN_MEMORY}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)


Train samples: 34163
Val samples  : 3795
DataLoader num_workers: 4
DataLoader pin_memory : False


## Step 4 — Verify Batch Shapes

Always check tensor shapes **before** training.  
A shape mismatch will cause a confusing error inside the model.

In [13]:
images, masks = next(iter(train_loader))
print(f"Image batch shape : {images.shape}   (expected [B, 3, H, W])")
print(f"Mask  batch shape : {masks.shape}    (expected [B, H, W])")
print(f"Image dtype       : {images.dtype}   (expected float32)")
print(f"Mask  dtype       : {masks.dtype}    (expected int64 / long)")
print(f"Image value range : [{images.min():.3f}, {images.max():.3f}]  (expected [0, 1])")

Image batch shape : torch.Size([4, 3, 256, 256])   (expected [B, 3, H, W])
Mask  batch shape : torch.Size([4, 256, 256])    (expected [B, H, W])
Image dtype       : torch.float32   (expected float32)
Mask  dtype       : torch.int64    (expected int64 / long)
Image value range : [0.008, 0.988]  (expected [0, 1])


## Step 5 — Define a Stronger Pretrained Segmentation Model

For the first real baseline, use a pretrained U-Net encoder so the model starts with useful low-level visual features (edges, textures) instead of learning everything from scratch.

In [9]:
import segmentation_models_pytorch as smp

# Use a lighter encoder on CPU to keep iterations practical.
device = get_device()
encoder_name = "mobilenet_v2" if device.type == "cpu" else "resnet34"

model = smp.Unet(
    encoder_name=encoder_name,
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(device)

# Quick shape check before training
dummy = torch.randn(2, 3, IMAGE_SIZE[1], IMAGE_SIZE[0]).to(device)
out = model(dummy)
print(f"Encoder: {encoder_name}")
print(f"Model output shape: {out.shape}   (expected [2, {NUM_CLASSES}, {IMAGE_SIZE[1]}, {IMAGE_SIZE[0]}])")

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

Using device: cpu


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

c:\Users\Jacob\AI4Mars\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jacob\.cache\huggingface\hub\models--smp-hub--mobilenet_v2.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

Encoder: mobilenet_v2
Model output shape: torch.Size([2, 4, 256, 256])   (expected [2, 4, 256, 256])
Model parameters: 6,629,380


## Step 6 — Loss Function and Optimiser

In [10]:
# Weighted cross-entropy to counter class imbalance.
# Approximation: soil common (lower weight), big_rock rare (higher weight).
weights = torch.tensor([1.0, 2.5, 1.5, 4.0], dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=IGNORE_INDEX)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(f"Class weights: {weights.tolist()}")

Class weights: [1.0, 2.5, 1.5, 4.0]


## Step 7 — Training Loop

In [ ]:
CLASS_NAMES = ["soil", "bedrock", "sand", "big_rock"]

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 40)

    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_metrics = evaluate(
        model,
        val_loader,
        loss_fn,
        device,
        num_classes=NUM_CLASSES,
        ignore_index=IGNORE_INDEX,
        return_per_class_iou=(epoch == NUM_EPOCHS),  # avoid extra overhead each epoch
    )

    print(f"  Train loss : {train_loss:.4f}")
    print(f"  Val loss   : {val_metrics['val_loss']:.4f}")
    print(f"  Pixel acc  : {val_metrics['pixel_acc']:.4f}")
    print(f"  Mean IoU   : {val_metrics['mean_iou']:.4f}")

    per_class_iou = val_metrics.get("per_class_iou")
    if per_class_iou is not None:
        print("  Per-class IoU:")
        valid_scores = []
        for class_name, score in zip(CLASS_NAMES, per_class_iou):
            if score is None:
                print(f"    - {class_name:8s}: n/a")
            else:
                print(f"    - {class_name:8s}: {score:.4f}")
                valid_scores.append((class_name, score))

        if valid_scores:
            hardest_class, hardest_score = min(valid_scores, key=lambda x: x[1])
            print(f"  Hardest class this epoch: {hardest_class} (IoU={hardest_score:.4f})")


Epoch 1/3
----------------------------------------
  Batch 10/8541  loss=1.2076
  Batch 20/8541  loss=1.4926
  Batch 30/8541  loss=1.1971
  Batch 40/8541  loss=1.3252
  Batch 50/8541  loss=1.3299
  Batch 60/8541  loss=0.8645
  Batch 70/8541  loss=0.7030
  Batch 80/8541  loss=0.8561
  Batch 90/8541  loss=0.8098
  Batch 100/8541  loss=1.8687
  Batch 110/8541  loss=0.9327
  Batch 120/8541  loss=1.7245
  Batch 130/8541  loss=1.1791
  Batch 140/8541  loss=0.8233
  Batch 150/8541  loss=0.9252
  Batch 160/8541  loss=1.0672
  Batch 170/8541  loss=1.6458
  Batch 180/8541  loss=1.1317
  Batch 190/8541  loss=1.0127
  Batch 200/8541  loss=1.6762
  Batch 210/8541  loss=0.7575
  Batch 220/8541  loss=0.8896
  Batch 230/8541  loss=1.7316
  Batch 240/8541  loss=1.2295
  Batch 250/8541  loss=1.0786
  Batch 260/8541  loss=1.4335
  Batch 270/8541  loss=1.4072
  Batch 280/8541  loss=1.6744
  Batch 290/8541  loss=1.0868
  Batch 300/8541  loss=1.2239
  Batch 310/8541  loss=1.1821
  Batch 320/8541  loss=0.83

KeyboardInterrupt: 

## Step 8 — Save Checkpoint

In [ ]:
checkpoint_path = MODELS_DIR / "weighted_unet_epoch03.pth"
save_checkpoint(model, optimizer, epoch=NUM_EPOCHS, path=checkpoint_path)
print(f"Saved checkpoint to {checkpoint_path}")

Checkpoint saved → C:\Users\Jacob\AI4Mars\models\baseline_epoch01.pth
Saved checkpoint to C:\Users\Jacob\AI4Mars\models\baseline_epoch01.pth


## Next Steps

- Open `04_evaluation_error_analysis.ipynb` to load the checkpoint and analyse predictions.
- To improve the model, replace `SimpleSegNet` with:
  - **U-Net** — classic encoder-decoder with skip connections.
  - **FCN** — `torchvision.models.segmentation.fcn_resnet50`.
  - **DeepLabv3** — `torchvision.models.segmentation.deeplabv3_resnet50`.